# Instance Segmentation
Today we will cover:
- Theory: Mask R-CNN
- Exercise: Segment multiple objects
- Mini Project: People segmentation in crowd images

## Theory: Mask R-CNN

- Extension of Faster R-CNN for instance segmentation.
- Adds a branch for predicting segmentation masks on each Region of Interest (RoI).
- Outputs: bounding boxes, class labels, and pixel-level masks.
- Key Idea: Detect objects and generate masks simultaneously.
- Applications: medical imaging, autonomous driving, crowd analysis.

## Setup: COCO Dataset Loader (for segmentation)

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CocoDetection

# Example COCO dataset path (replace with actual path)
coco_root = './coco_data'

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

trainset = CocoDetection(root=coco_root+'/train2017', annFile=coco_root+'/annotations/instances_train2017.json', transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=2, shuffle=True)

print("Loaded COCO dataset with", len(trainset), "images")

## Exercise: Segment Multiple Objects with Mask R-CNN

In [ ]:
import torchvision.models.detection as detection
import matplotlib.pyplot as plt

# Load pretrained Mask R-CNN model
model = detection.maskrcnn_resnet50_fpn(pretrained=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Run segmentation on a batch
images, targets = next(iter(trainloader))
images = [img.to(device) for img in images]
outputs = model(images)

print("Segmented objects in batch:")
for i, output in enumerate(outputs):
    print(f"Image {i}: {len(output['masks'])} masks generated")

## Mini Project: People Segmentation in Crowd Images

In [ ]:
import cv2
from PIL import Image
import numpy as np

# Load a sample crowd image (replace with actual path)
crowd_image = Image.open('crowd.jpg').convert("RGB")
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])
input_tensor = transform(crowd_image).unsqueeze(0).to(device)

# Run Mask R-CNN
output = model(input_tensor)[0]

# Extract masks for 'person' class (COCO class id = 1)
person_masks = []
for mask, label in zip(output['masks'], output['labels']):
    if label.item() == 1:  # person class
        person_masks.append(mask.squeeze().cpu().detach().numpy())

# Visualize first mask
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(crowd_image)
plt.title('Original Crowd Image')
plt.subplot(1,2,2)
plt.imshow(person_masks[0], cmap='gray')
plt.title('Person Mask')
plt.show()